In [1]:
import os
os.chdir('application')

In [2]:
import uuid

from typing import List
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import redis

from modules.redis import RedisConversationMemory
# from modules.retriever_wrappper import SanitizingRetriever
from modules.history_utils import build_chat_history_messages

from config import (REDIS_URL, SESSION_TTL_SECONDS, MAX_HISTORY_MESSAGES, 
                    gemini_chat_model, api_key_name, system_prompt, qa_prompt
)
from vectorstore.milvus_vectorstore import get_vectorstore

import logging

# LangChain imports (use provider wrapper)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

C:\Users\USER\Miniconda3\envs\kpn-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

In [4]:
# ----- CONFIG -----
r = redis.Redis.from_url(REDIS_URL, decode_responses=True)
# app = FastAPI()

In [5]:
# Your already-created vectorstore (LangChain Milvus wrapper)
vectorstore = get_vectorstore()

In [6]:
mem = RedisConversationMemory(session_id='001', max_messages=MAX_HISTORY_MESSAGES)


In [7]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})

In [8]:
llm = ChatGoogleGenerativeAI(model=gemini_chat_model, google_api_key=os.environ.get(api_key_name))

In [9]:
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_prompt),
    MessagesPlaceholder(variable_name="chat_history"),            # inject conversation history here
    ("system", "Gunakan dokumen yang diekstrak berikut untuk menjawab pertanyaan user:\n\n{context}"),
    ("human", "{question}")
])

In [10]:
def format_docs(docs: List) -> str:
    """Helper function to combine document contents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

In [11]:
rag_chain = (
    {
        # The retriever runs first, using the original question.
        # Its output is passed to format_docs and becomes the 'context'.
        "context": (lambda x: x["question"]) | retriever | format_docs,
        
        # The original 'question' is passed through unchanged.
        "question": (lambda x: x["question"]),
        
        # The 'chat_history' is passed through unchanged.
        "chat_history": (lambda x: x["chat_history"])
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

In [12]:
raw_history = mem.get_chat_history_pairs() 
raw_history

[]

In [13]:
question = 'Bagaimana jika saya ingin cuti karena sakit?'

In [14]:
chat_history_messages = build_chat_history_messages(raw_history)
chat_history_messages

Raw chat history input: []
"raw is None" case


[HumanMessage(content='', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={})]

In [15]:
# The invoke call now passes a single dictionary with all the required keys
result = rag_chain.invoke({
    "question": question, 
    "chat_history": chat_history_messages
})

In [16]:
result

'Mohon maaf, berdasarkan dokumen yang saya miliki, tidak ada informasi spesifik mengenai "Cuti Sakit" untuk karyawan yang sakit.\n\nDokumen ini hanya menyebutkan:\n*   **Cuti Pinjam** dapat diajukan untuk Keadaan Mendesak seperti "Keluarga Inti sakit" (Sumber: Baris 77).\n*   **Cuti Tidak Dibayar** dapat diajukan apabila "Harus merawat Keluarga Inti yang sakit" (Sumber: Baris 145).\n\nUntuk pertanyaan mengenai cuti karena Anda sendiri sakit, saya sarankan Anda untuk menghubungi tim HRD KPN-Corp secara langsung untuk mendapatkan informasi lebih lanjut mengenai kebijakan perusahaan terkait hal ini.'

In [17]:
stop

NameError: name 'stop' is not defined

In [ ]:
conv_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=None,  # ok if you pass chat_history manually
    return_source_documents=True,
    combine_docs_chain_kwargs={"prompt": qa_prompt},
)

In [ ]:
result = conv_chain.invoke({"question": question, "chat_history": chat_history_messages})
result

In [ ]:
from langchain.chains import (
    create_history_aware_retriever,
    create_retrieval_chain,
)
from langchain.chains.combine_documents import create_stuff_documents_chain

In [ ]:
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
# contextualize_q_prompt

In [ ]:
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [ ]:
qa_prompt.input_variables

In [ ]:
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)